# Greenhouse problem with Additional Greenhouse Space Option:   
## How best to allocated a Greenhouse to different produce?

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2024, University of Chicago 

---

- You have a new start-up that grows up to 4 types (A, B, C, D) of produce in an organic greenhouse that you then sell to local restaurants.
- Consider this “plant portfolio” optimization problem.
- Each plant type is sold and planted in units called “containers”.  Each plant container begins as a seedling.
A container of each type of produce may occupy a different amount of physical space (measured in square meters) in your greenhouse:  Sa, Sb, Sc, Sd.   You have a total of S square meters of growing space in your greenhouse.
- At harvest, each container has an expected yield, selling price, and cost to purchase and maintain, and so results in a profit per container:  Pa, Pb, Pc, Pd.
- Each container requires some Time (number of days) to grow from seedling to full plant, ready to harvest: Ta, Tb, Tc, Td
- Assume that at harvest a container of a particular plant is immediately replaced by a new container of the same plant.  So we always have the same number of containers of each plant type in our Greenhouse.
- In order to meet demand and bundling obligations we have the following conditions to meet:
  - The number of containers of A must be at least twice the containers of B harvested each day, on average.
  - The daily containers of B harvested must be at least 20% of all containers harvested each day, on average.
- Formulate an LP to determine a good long run plant mix strategy.  That is, how many containers of each type should be in the greenhouse?  Assume you can sell anything you can feasibly produce.  HINT:  Your objective function should seek to Maximize the average profits per day.  To simplify our model, we are not concerned that some days we will harvest more than others, instead we pretend harvesting (and thus profits) are smooth by considering the long term average yield from each container each day.

### New Additional Greenhouse Space available
- Greenhouse 1 with S1 space for daily cost C1 
- Greenhouse 2 with S2 space for daily cost C2 
- Greehouse 2 is an economical extension of Greenhouse 1, but can only be used if Greenhouse 1 is used.

### Data
- S = 10000
- S1 = 3000
- S2 = 3500
- C1 = 125
- C2 = 100

| &nbsp;    |  S(pace) |  P(rofit)  |  T(ime) |  
| ----------|     ----:|        ---:|     ---:| 
| A         |    50    |    20      |  20     |
| B         |    80    |    100      |  40     | 
| C         |    14    |    40      |  10     | 
| D         |    33    |    80      |  32     |



### Linear Program

- Variables
  - A, B, C, D: number of containers in greenhouse of each type of produce 
  - G1, G2 : Binary if Greenhouse built
 
- Constants
  - Sa, Sb, Sc, Sd: Space (square meters) per container
  - Pa, Pb, Pc, Pd: Profit per container
  - Ta, Tb, Tc, Td: Number of days required to harvest each type of produce 
  - C1, C2:  Daily cost of additional greenhouses
  - S1, S2:  Space (square meters) of additional greenhouses
  
- Formulation
  - Maximize Pa/Ta * A + Pb/Tb * B + Pc/Tc * C + Pd/Td * D - C1 G1 - C2 G2
  - Subject to:
    - Sa A + Sb B + Sc C + Sd D  <= S + S1 G1 + S2 G2
    - A/Ta >= 2 \* B/Tb 
    - B/Tb >= 0.20 (A/Ta + B/Tb + C/Tc + D/Td)

In [1]:
import pulp 

In [2]:
from pprint import pprint

In [3]:
import pulp
import os

# Portable solver setup, to allow work locally (ARM64 architecture) as well as in JupyterHub
from pulp import COIN_CMD
if os.path.exists('/opt/homebrew/opt/cbc/bin/cbc'):
    solver = COIN_CMD(path='/opt/homebrew/opt/cbc/bin/cbc', msg=0)
else:
    solver = pulp.PULP_CBC_CMD(msg=0)

### Load data in Python variables

In [4]:
S = 10000

S1 = 3000
S2 = 3500
C1 = 125
C2 = 100 

space = {}
space['A'] = 50 
space['B'] = 80 
space['C'] = 14 
space['D'] = 33 

profit = {'A': 20, 'B': 100, 'C': 40, 'D': 80}
time = {'A': 20, 'B': 40, 'C': 10, 'D': 32}

In [5]:
model = pulp.LpProblem("Greenhouse_Strategy",pulp.LpMaximize)

### Define 6 decision variables

In [6]:
A =  pulp.LpVariable("A", lowBound = 0, cat='Integer')
B =  pulp.LpVariable("B", lowBound = 0, cat='Integer')
C =  pulp.LpVariable("C", lowBound = 0, cat='Integer')
D =  pulp.LpVariable("D", lowBound = 0, cat='Integer')
G1 = pulp.LpVariable("G_1_boolean", cat='Binary')
G2 = pulp.LpVariable("G_2_boolean", cat='Binary')

In [7]:
## NOTE:  We used the Python line continuation character "\".  
##        Make sure you have nothing, even spaces after it
model += (profit['A']/time['A'])*A + (profit['B']/time['B']*B) + \
         (profit['C']/time['C'])*C + (profit['D']/time['D'])*D - C1 * G1 - C2 * G2
print(model)

Greenhouse_Strategy:
MAXIMIZE
1.0*A + 2.5*B + 4.0*C + 2.5*D + -125*G_1_boolean + -100*G_2_boolean + 0.0
VARIABLES
0 <= A Integer
0 <= B Integer
0 <= C Integer
0 <= D Integer
0 <= G_1_boolean <= 1 Integer
0 <= G_2_boolean <= 1 Integer



### Add constraints

In [8]:
# We can first load the "body" of the constraint
const = space['A']*A + space['B']*B + space['C']*C + space['D']*D

# now "add" the constraint to the model with the RHS.  And name it. 
model += const <= S + S1 * G1 + S2 * G2, "Space_Constraint" 

In [9]:
print(model)

Greenhouse_Strategy:
MAXIMIZE
1.0*A + 2.5*B + 4.0*C + 2.5*D + -125*G_1_boolean + -100*G_2_boolean + 0.0
SUBJECT TO
Space_Constraint: 50 A + 80 B + 14 C + 33 D - 3000 G_1_boolean
 - 3500 G_2_boolean <= 10000

VARIABLES
0 <= A Integer
0 <= B Integer
0 <= C Integer
0 <= D Integer
0 <= G_1_boolean <= 1 Integer
0 <= G_2_boolean <= 1 Integer



In [10]:
# A-B constraint
# NOTE:  writing A/time['A'] does not work !!  
#
# Here we put the constraint directly into the model
model += (1.0/time['A'])*A  >=  (2.0/time['B'])*B, "A_B"

In [11]:
# B tot constraint
#
model += (1.0/time['B'])*B  >=  \
         0.20*( (1.0/time['A'])*A + (1.0/time['B'])*B + (1.0/time['C'])*C + (1.0/time['D'])*D), "B_tot"
print(model)

Greenhouse_Strategy:
MAXIMIZE
1.0*A + 2.5*B + 4.0*C + 2.5*D + -125*G_1_boolean + -100*G_2_boolean + 0.0
SUBJECT TO
Space_Constraint: 50 A + 80 B + 14 C + 33 D - 3000 G_1_boolean
 - 3500 G_2_boolean <= 10000

A_B: 0.05 A - 0.05 B >= 0

B_tot: - 0.01 A + 0.02 B - 0.02 C - 0.00625 D >= 0

VARIABLES
0 <= A Integer
0 <= B Integer
0 <= C Integer
0 <= D Integer
0 <= G_1_boolean <= 1 Integer
0 <= G_2_boolean <= 1 Integer



In [12]:
model += G2 <= G1, "G1G2_Sequence_Constraint"

In [13]:
# Let's solve the model and make sure it's status is good
model.solve(solver)
print("Status:", pulp.LpStatus[model.status])

Status: Optimal


In [14]:
print("Total Objective function value", pulp.value(model.objective))

Total Objective function value 450.0


In [15]:
# Here is the solution
for v in model.variables():
    print(v.name, "=", v.varValue)

A = 90.0
B = 90.0
C = 0.0
D = 144.0
G_1_boolean = 1.0
G_2_boolean = 1.0
